# E-commerce Customer Purchase Behavior Analysis & Customer Segmentation

## Business Problem

This project analyzes e-commerce transaction data to understand customer purchase behavior, segment customers based on purchasing patterns, and generate business recommendations for customer retention, targeting, and cross-selling strategies.

The main business questions are:

1. Who are the most valuable customers?
2. Which customers are at risk of churn?
3. What products generate the most revenue?
4. What products are frequently bought together?
5. Can customers be grouped into meaningful segments?
6. What marketing strategies should be applied to each segment?

In [1]:
import os
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

In [3]:
BASE_DIR = os.path.dirname(os.getcwd())
RAW_DATA_PATH = os.path.join(BASE_DIR, "data", "raw", "online_retail_II.xlsx")

print("BASE_DIR:", BASE_DIR)
print("RAW_DATA_PATH:", RAW_DATA_PATH)

BASE_DIR: D:\AI-Learning\ecommerce-customer-segmentation
RAW_DATA_PATH: D:\AI-Learning\ecommerce-customer-segmentation\data\raw\online_retail_II.xlsx


In [5]:
df = pd.read_excel(RAW_DATA_PATH)
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [7]:
df.shape

(525461, 8)

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 525461 entries, 0 to 525460
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      525461 non-null  object        
 1   StockCode    525461 non-null  object        
 2   Description  522533 non-null  object        
 3   Quantity     525461 non-null  int64         
 4   InvoiceDate  525461 non-null  datetime64[ns]
 5   Price        525461 non-null  float64       
 6   Customer ID  417534 non-null  float64       
 7   Country      525461 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 32.1+ MB


In [9]:
df.isna().sum()

Invoice             0
StockCode           0
Description      2928
Quantity            0
InvoiceDate         0
Price               0
Customer ID    107927
Country             0
dtype: int64

In [10]:
df.describe()

,Quantity,InvoiceDate,Price,Customer ID
count,525461.000000,525461,525461.000000,417534.000000
mean,10.337667,2010-06-28 11:37:36.845017856,4.688834,15360.645478
min,-9600.000000,2009-12-01 07:45:00,-53594.360000,12346.000000
25%,1.000000,2010-03-21 12:20:00,1.250000,13983.000000
50%,3.000000,2010-07-06 09:51:00,2.100000,15311.000000
75%,10.000000,2010-10-15 12:45:00,4.210000,16799.000000
max,19152.000000,2010-12-09 20:01:00,25111.090000,18287.000000
std,107.424110,NaN,146.126914,1680.811316


## Dataset Columns

- InvoiceNo: Mã hóa đơn. Nếu bắt đầu bằng chữ C, đó có thể là hóa đơn hủy.
- StockCode: Mã sản phẩm.
- Description: Tên sản phẩm.
- Quantity: Số lượng sản phẩm trong dòng giao dịch.
- InvoiceDate: Ngày và giờ giao dịch.
- UnitPrice: Giá mỗi đơn vị sản phẩm.
- CustomerID: Mã khách hàng.
- Country: Quốc gia của khách hàng.

## Research Questions

### Sales Behavior
1. What is the total revenue?
2. How does revenue change over time?
3. Which products generate the highest revenue?
4. Which products are sold the most?

### Customer Behavior
1. How many unique customers are there?
2. Which customers generate the highest revenue?
3. How often do customers purchase?
4. What is the average order value?

### Geographic Behavior
1. Which countries generate the most revenue?
2. Is the business concentrated in one country?

### Segmentation
1. Can customers be segmented by Recency, Frequency, and Monetary value?
2. What are the characteristics of each customer segment?
3. What business actions should be recommended for each segment?

In [11]:
df_raw = df.copy()

print("Raw data shape:", df_raw.shape)
df_raw.head()

Raw data shape: (525461, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [14]:
missing_values = df_raw.isna().sum()
missing_values

missing_percentage = df_raw.isna().mean() * 100
missing_percentage.sort_values(ascending=False)

Customer ID    20.539488
Description     0.557225
StockCode       0.000000
Invoice         0.000000
Quantity        0.000000
InvoiceDate     0.000000
Price           0.000000
Country         0.000000
dtype: float64